# Colorizar una foto familiar en blanco y negro

Este cuaderno construye el MVP paso a paso: una persona de la familia carga una foto, un modelo de difusión propone colores, CLIP compara la imagen antes y después y Gradio muestra ambas versiones.

> **Importante:** una foto en blanco y negro no contiene sus colores originales. El modelo genera una propuesta verosímil, no recupera ni verifica la verdad histórica. Conservá el original.

El trabajo usa un ControlNet de Hugging Face y un modelo base Stable Diffusion 2.1. La ficha de ControlNet solo declara licencia OpenRAIL y casi no documenta sus datos o resultados; lo probaremos como experimento, no como una solución validada. Sus pesos ocupan alrededor de 1,46 GB, además de los varios GB del modelo base.

## Antes de empezar

- Seleccioná el kernel **Python 3.12 (colorizador-fotos)**; usa el único entorno `.venv` del proyecto.
- Hace falta Internet para descargar los modelos la primera vez.
- Los modelos seleccionados son públicos y no requieren autenticación. Sus licencias OpenRAIL igualmente tienen condiciones de uso.
- Esta computadora tiene PyTorch CPU, sin CUDA. La inferencia puede tardar mucho; para una GPU se puede adaptar el entorno en Colab. No subas fotos familiares privadas a un entorno remoto sin permiso.

Recorrido: rutas y dispositivo → modelos → preparación de imagen → colorización → comparación y CLIP → interfaz local → exportación opcional de `app.py`.

## 1. Preparar las carpetas

Guardamos cachés y temporales dentro del proyecto en E:. Así se evita llenar la unidad del sistema. Ejecutá esta celda antes de importar Diffusers o Transformers.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path(r"E:\ifts24-lab-pdi-2026\TP2")

CACHE_DIR = ROOT / ".model-cache"
HF_HOME = CACHE_DIR / "huggingface"
OUTPUT_DIR = CACHE_DIR / "resultados"
for carpeta in (HF_HOME, OUTPUT_DIR, CACHE_DIR / "torch", CACHE_DIR / "gradio"):
    carpeta.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["GRADIO_TEMP_DIR"] = str(CACHE_DIR / "gradio")
os.environ["TEMP"] = str(CACHE_DIR)
os.environ["TMP"] = str(CACHE_DIR)

print("Proyecto:", ROOT)
print("Caché de modelos:", HF_HOME)
print("Resultados:", OUTPUT_DIR)

Proyecto: E:\ifts24-lab-pdi-2026\TP2
Caché de modelos: E:\ifts24-lab-pdi-2026\TP2/.model-cache/huggingface
Resultados: E:\ifts24-lab-pdi-2026\TP2/.model-cache/resultados


## 2. Comprobar el entorno

Elegimos automáticamente GPU si PyTorch la detecta; si no, usamos CPU. En CPU se conserva precisión completa. No se fuerza una descarga ni se ejecuta todavía el modelo.

In [2]:
import torch
from PIL import Image, ImageOps
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: la difusión en CPU puede tardar mucho; usá una foto pequeña para la primera prueba.")

PyTorch: 2.14.1+cu130
Dispositivo: cuda
GPU: Tesla T4


## 3. Modelos públicos de Hugging Face

El modelo base [`sd2-community/stable-diffusion-2-1-base`](https://huggingface.co/sd2-community/stable-diffusion-2-1-base) y el ControlNet de colorización son públicos en Hugging Face: no hace falta iniciar sesión ni ingresar un token. Ambos publican sus pesos bajo licencias OpenRAIL; revisá sus condiciones de uso. El repositorio base informa la arquitectura Stable Diffusion 2.1, compatible con el ControlNet elegido.

## 4. Cargar el modelo de difusión

ControlNet recibe la estructura de la foto en escala de grises y condiciona la generación. El modelo base propone los colores. Fijamos las revisiones de ambos repositorios para que la descarga sea reproducible. La primera descarga del modelo base ocupa varios GB.

In [3]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

CONTROLNET_ID = "neurallove/controlnet-sd21-colorization-diffusers"
CONTROLNET_REVISION = "2083978ca9345c9edac3af2779c17710b7e42258"
BASE_MODEL_ID = "sd2-community/stable-diffusion-2-1-base"
BASE_MODEL_REVISION = "4e63672c03103b6c636b8fb4119ba982469b2955"

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID,
    revision=CONTROLNET_REVISION,
    use_safetensors=False,
    torch_dtype=DTYPE,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    controlnet=controlnet,
    torch_dtype=DTYPE,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=False)

PIPELINE_LISTO = True
print("Modelo de difusión cargado en", DEVICE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

diffusion_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.46GB            

diffusion_pytorch_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Modelo de difusión cargado en cuda


## 5. Preparar una imagen sin deformarla

Convertimos la entrada a blanco y negro para que el experimento sea consistente. Reducimos el lado mayor a 512 píxeles y redondeamos dimensiones a múltiplos de 8, que son adecuados para la arquitectura de Stable Diffusion. El resultado se ampliará a las dimensiones originales para poder compararlo.

In [4]:
def preparar_imagen(imagen, lado_maximo=512):
    """Devuelve la foto gris y su versión reducida para el modelo."""
    original = ImageOps.grayscale(imagen).convert("RGB")
    ancho, alto = original.size
    escala = min(1.0, lado_maximo / max(ancho, alto))
    nuevo_ancho = max(64, int(ancho * escala) // 8 * 8)
    nuevo_alto = max(64, int(alto * escala) // 8 * 8)
    reducida = original.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    return original, reducida

print("Función de preparación lista. La interfaz permitirá cargar una imagen cuando esté completa.")

Función de preparación lista. La interfaz permitirá cargar una imagen cuando esté completa.


## 6. Generar la propuesta de color

La semilla hace que una repetición con los mismos parámetros produzca un resultado más comparable. El prompt orienta el estilo; no aporta información verdadera sobre los colores de la escena. Bajá los pasos para una prueba rápida, sabiendo que eso también puede cambiar el resultado.

In [5]:
PROMPT = "colorized vintage black and white family photograph, natural realistic colors, preserve the original composition and details"
NEGATIVE_PROMPT = "black and white, grayscale, sepia, changed composition, distorted face, artifacts, text"

def colorizar(imagen, semilla=42, pasos=15):
    if "pipe" not in globals():
        raise RuntimeError("Primero ejecutá la celda que carga el modelo de difusión.")
    original, control = preparar_imagen(imagen)
    generador = torch.Generator(device=DEVICE).manual_seed(int(semilla))
    resultado = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=control,
        width=control.width,
        height=control.height,
        num_inference_steps=int(pasos),
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        generator=generador,
    ).images[0]
    resultado = resultado.resize(original.size, Image.Resampling.LANCZOS)
    return original, resultado

print("Función de colorización lista; todavía no se descargó ni ejecutó ninguna foto.")

Función de colorización lista; todavía no se descargó ni ejecutó ninguna foto.


## 7. Preparar el análisis visual con CLIP

CLIP compara la imagen con dos descripciones de texto. Sus puntajes expresan afinidad relativa con esas etiquetas: **no** miden calidad, naturalidad, identidad ni autenticidad de los colores. Cargamos este segundo modelo desde Hugging Face; también descarga sus propios pesos.

In [6]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, torch_dtype=DTYPE).to(DEVICE).eval()
CLIP_LISTO = True
print("CLIP cargado en", DEVICE)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP cargado en cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

In [7]:
ETIQUETAS_CLIP = ["a black and white photograph", "a color photograph"]

@torch.inference_mode()
def analizar_con_clip(imagenes):
    entradas = clip_processor(
        text=ETIQUETAS_CLIP,
        images=imagenes,
        return_tensors="pt",
        padding=True,
    )
    entradas = {nombre: tensor.to(DEVICE) for nombre, tensor in entradas.items()}
    puntajes = clip_model(**entradas).logits_per_image.softmax(dim=1).cpu().tolist()
    return puntajes

print("Analizador CLIP listo.")

Analizador CLIP listo.


## 8. Preparar la comparación

Esta función reúne procesamiento y análisis para que la use tanto el cuaderno como Gradio. En la tabla veremos qué etiqueta se parece más a cada imagen; el puntaje no sirve como verificación del color real.

In [8]:
PRUEBA_OK = False

def procesar_foto(imagen, semilla=42, pasos=15):
    global PRUEBA_OK
    if imagen is None:
        raise ValueError("Cargá una imagen para comenzar.")
    original, colorizada = colorizar(imagen, semilla=semilla, pasos=pasos)
    puntajes = analizar_con_clip([original, colorizada])
    nombres = ["Original", "Colorizada"]
    reporte = "| Imagen | Blanco y negro | A color |\n|---|---:|---:|\n"
    for nombre, fila in zip(nombres, puntajes):
        reporte += f"| {nombre} | {fila[0]:.1%} | {fila[1]:.1%} |\n"
    reporte += "\n*CLIP estima afinidad con dos etiquetas; no comprueba los colores originales.*"
    PRUEBA_OK = True
    return original, colorizada, reporte

print("Flujo integrado listo. La primera prueba real se hará al cargar una foto en la interfaz.")

Flujo integrado listo. La primera prueba real se hará al cargar una foto en la interfaz.


## 9. Abrir la interfaz desde este cuaderno

La app corre localmente desde el kernel. `share=False` evita crear un enlace público. Cargá una foto no sensible para la primera prueba; la comparación y el aviso aparecen en la misma pantalla.

In [9]:
import gradio as gr

with gr.Blocks(title="Colorizador de fotos familiares") as demo:
    gr.Markdown(
        "# Colorizador de fotos familiares\n"
        "**Los colores son una propuesta de IA, no los colores históricos comprobados. Conservá tu original.**"
    )
    with gr.Row():
        entrada = gr.Image(label="Foto en blanco y negro", type="pil")
        original_salida = gr.Image(label="Original en escala de grises", type="pil")
        color_salida = gr.Image(label="Propuesta colorizada", type="pil")
    pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión (más pasos pueden tardar más)")
    semilla = gr.Number(value=42, precision=0, label="Semilla")
    boton = gr.Button("Colorizar y analizar")
    analisis = gr.Markdown()
    boton.click(
        fn=procesar_foto,
        inputs=[entrada, semilla, pasos],
        outputs=[original_salida, color_salida, analisis],
    )

demo.launch(share=False, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [10]:
import gradio as gr

with gr.Blocks(title="Colorizador de fotos familiares") as demo:
    gr.Markdown(
        "# Colorizador de fotos familiares"   
    )
    with gr.Row():
        entrada = gr.Image(label="Foto en blanco y negro", type="pil")
        original_salida = gr.Image(label="Original, escala de grises", type="pil")
        color_salida = gr.Image(label="Propuesta colorizada", type="pil")
    pasos = gr.Slider(5, 25, value=15, step=1, label="Pasos de difusión (más pasos pueden tardar más)")
    semilla = gr.Number(value=42, precision=0, label="Semilla")
    boton = gr.Button("Colorizar y analizar")
    analisis = gr.Markdown()
    boton.click(
        fn=procesar_foto,
        inputs=[entrada, semilla, pasos],
        outputs=[original_salida, color_salida, analisis],
    )

demo.launch(share=False, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 10. Qué observar y explicar

1. ¿Qué formas y tonos de la foto original se conservaron?
2. ¿Qué zonas recibieron colores que no podemos comprobar?
3. ¿Cambió CLIP su afinidad hacia “a color”? Eso describe la etiqueta, no la calidad.
4. ¿Qué pasa al repetir con otra semilla o cantidad de pasos?

La foto no se envía a una API de pago ni se publica mediante un enlace compartido. El modelo agrega contenido plausible y puede alterar detalles; compará siempre con el original.